In [2]:
# train_model_pycaret_max.py
# ==========================================================
# PIPELINE COMPLETO - Wine Quality Classification (MAX PyCaret)
# Dataset: https://www.kaggle.com/datasets/yasserh/wine-quality-dataset
#
# ✔ EDA com ydata-profiling
# ✔ Balanceamento (gráfico)
# ✔ Feature Engineering manual
# ✔ PyCaret setup com pré-processamentos avançados
# ✔ compare_models com turbo=False (máximo de modelos possíveis)
# ✔ tuning do Top-N
# ✔ tentativa de ensemble (blend/stack) quando disponível
# ✔ finalize e save_model (pipeline completo)
# ✔ salva tudo dentro de wine_pycaret/
#
# ✅ Correção para Streamlit único:
# - meta.json passa a ter:
#   "model_kind": "pycaret"
#   "artifact_path": "model_pipeline"  (prefixo sem .pkl)
# ==========================================================

import os
import json
import glob
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ydata_profiling import ProfileReport
import kagglehub

from pycaret.classification import ClassificationExperiment

# =========================
# Configurações
# =========================
RANDOM_STATE = 42

BASE_DIR = os.path.abspath("wine_pycaret")
DIR_REPORTS = os.path.join(BASE_DIR, "reports")
DIR_FIGURES = os.path.join(BASE_DIR, "figures")

# ✅ Cria a base primeiro (garante que apareça no lugar certo)
os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(DIR_REPORTS, exist_ok=True)
os.makedirs(DIR_FIGURES, exist_ok=True)

print("Diretório base usado:", BASE_DIR)

# =========================
# Funções auxiliares
# =========================
def load_dataset():
    """Baixa e carrega o dataset via kagglehub."""
    path = kagglehub.dataset_download("yasserh/wine-quality-dataset")
    csv = [c for c in glob.glob(os.path.join(path, "*.csv")) if "WineQT" in c][0]
    df = pd.read_csv(csv)
    if "Id" in df.columns:
        df = df.drop(columns=["Id"])
    return df


def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """Cria features adicionais (didáticas)."""
    eps = 1e-9
    out = df.copy()

    out["total_acidity"] = (
        out["fixed acidity"]
        + out["volatile acidity"]
        + out["citric acid"]
    )

    out["alcohol_sugar_ratio"] = out["alcohol"] / (out["residual sugar"] + eps)

    out["so2_ratio"] = (
        out["free sulfur dioxide"] /
        (out["total sulfur dioxide"] + eps)
    )

    out["density_alcohol"] = out["density"] * out["alcohol"]
    return out


def plot_class_balance(y: pd.Series):
    plt.figure(figsize=(7, 4))
    sns.countplot(x=y, color="maroon")
    plt.title("Distribuição das Classes (quality)")
    plt.xlabel("Quality")
    plt.ylabel("Contagem")
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_FIGURES, "class_balance.png"), dpi=200)
    plt.close()


# =========================
# MAIN
# =========================
def main():
    print("[1/7] Carregando dataset...")
    df = load_dataset()
    print("Shape:", df.shape)

    print("[2/7] Gerando EDA (ydata-profiling)...")
    ProfileReport(df, title="Wine Quality - EDA (PyCaret MAX)", explorative=True)\
        .to_file(os.path.join(DIR_REPORTS, "eda_wine_quality.html"))

    # Gráfico de distribuição do target
    plot_class_balance(df["quality"].astype(int))

    print("[3/7] Feature Engineering...")
    df_fe = feature_engineering(df)

    # ✅ NORMALIZA nomes de colunas (EVITA ERRO DE FEATURE NAMES)
    df_fe.columns = (
        df_fe.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    print("\nDistribuição das classes:")
    print(df_fe["quality"].value_counts().sort_index())

    # =========================
    # PyCaret Setup (MAX)
    # =========================
    print("\n[4/7] Setup PyCaret (pré-processamento avançado)...")
    exp = ClassificationExperiment()

    exp.setup(
        data=df_fe,
        target="quality",
        session_id=RANDOM_STATE,

        # Split e CV
        fold_strategy="stratifiedkfold",
        fold=5,

        # Pré-processamento (mais completo)
        imputation_type="simple",
        normalize=True,
        normalize_method="zscore",
        transformation=True,
        transformation_method="yeo-johnson",

        # Outliers / multicolinearidade / seleção de features
        remove_outliers=True,
        outliers_method="iforest",
        outliers_threshold=0.03,              # ~3% (ajuste seguro p/ dataset pequeno)
        remove_multicollinearity=True,
        multicollinearity_threshold=0.9,
        feature_selection=True,
        feature_selection_method="classic",
        n_features_to_select=0.8,             # mantém 80% (evita cortar demais)

        # IMPORTANTÍSSIMO no WineQT multiclasses raro:
        # fix_imbalance pode causar distorção; deixamos desligado.
        fix_imbalance=False,

        # Logs/saída
        html=True,
        verbose=False
    )

    # =========================
    # Compare Models (MAX)
    # =========================
    print("\n[5/7] compare_models (turbo=False = máximo de modelos da biblioteca)...")
    top_models = exp.compare_models(
        sort="F1",
        n_select=5,
        turbo=False
    )

    # leaderboard
    leaderboard = exp.pull()
    leaderboard_path = os.path.join(BASE_DIR, "leaderboard_compare_models.csv")
    leaderboard.to_csv(leaderboard_path, index=False)
    print(f"[OK] Leaderboard salvo em: {leaderboard_path}")

    # top_models pode ser lista quando n_select > 1
    if not isinstance(top_models, list):
        top_models = [top_models]

    print("\n✅ Top modelos retornados:")
    for i, m in enumerate(top_models, 1):
        print(f"{i}. {m.__class__.__name__}")

    # =========================
    # Tuning do Top-N
    # =========================
    print("\n[6/7] Tuning automático dos Top modelos...")
    tuned_models = []
    for m in top_models:
        try:
            tuned = exp.tune_model(m, optimize="F1")
            tuned_models.append(tuned)
        except Exception as e:
            print(f"[AVISO] tune_model falhou para {m.__class__.__name__}: {e}")

    # Se não conseguiu tunar nada, use o melhor original
    candidates = tuned_models if tuned_models else top_models

    # =========================
    # (Opcional) Ensembles (Blend/Stack)
    # =========================
    ensemble_candidates = []
    if len(candidates) >= 2:
        try:
            blend = exp.blend_models(estimator_list=candidates, optimize="F1")
            ensemble_candidates.append(blend)
            print("[OK] Blend criado com sucesso.")
        except Exception as e:
            print(f"[INFO] blend_models indisponível/ falhou: {e}")

        try:
            stack = exp.stack_models(estimator_list=candidates, optimize="F1")
            ensemble_candidates.append(stack)
            print("[OK] Stack criado com sucesso.")
        except Exception as e:
            print(f"[INFO] stack_models indisponível/ falhou: {e}")

    final_pool = candidates + ensemble_candidates

    best_final = None
    best_score = -1.0

    # Avalia no holdout via predict_model (PyCaret aplica pipeline completo)
    for mdl in final_pool:
        try:
            _ = exp.predict_model(mdl)
            metrics = exp.pull()

            if "F1" in metrics.columns:
                score = float(metrics["F1"].values[0])
            elif "Accuracy" in metrics.columns:
                score = float(metrics["Accuracy"].values[0])
            else:
                score = 0.0

            if score > best_score:
                best_score = score
                best_final = mdl
        except Exception as e:
            print(f"[AVISO] Avaliação falhou para {mdl.__class__.__name__}: {e}")

    if best_final is None:
        best_final = candidates[0]

    print(f"\n✅ Melhor escolhido (holdout) = {best_final.__class__.__name__} | score={best_score:.4f}")

    # Finaliza (treina no dataset completo com o pipeline)
    final_model = exp.finalize_model(best_final)

    # =========================================================
    # ✅ EXPORTAÇÃO PADRÃO PARA STREAMLIT ÚNICO
    # =========================================================
    model_prefix = "model_pipeline"  # PyCaret cria model_pipeline.pkl
    exp.save_model(final_model, os.path.join(BASE_DIR, model_prefix))

    # Metadados (padronizados para Streamlit único)
    meta = {
        # contrato do Streamlit único
        "model_kind": "pycaret",
        "artifact_path": model_prefix,  # prefixo SEM .pkl e SEM pasta
        "target": "quality",
        "features": df_fe.drop(columns=["quality"]).columns.tolist(),
        "classes": sorted(df_fe["quality"].unique().tolist()),

        # extras (mantém seus dados)
        "pipeline_type": "pycaret_classification",
        "task": "multiclass_classification",
        "dataset": "Wine Quality (Kaggle - yasserh)",
        "random_state": RANDOM_STATE,
        "best_model_class": best_final.__class__.__name__,
        "best_holdout_score": best_score,
        "artifacts": {
            "model_pipeline_prefix": f"{BASE_DIR}/{model_prefix}",
            "leaderboard": f"{BASE_DIR}/leaderboard_compare_models.csv",
            "eda": f"{BASE_DIR}/reports/eda_wine_quality.html",
            "fig_balance": f"{BASE_DIR}/figures/class_balance.png"
        }
    }

    with open(os.path.join(BASE_DIR, "meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print("\n✅ PIPELINE FINALIZADO COM SUCESSO")
    print("Estrutura gerada dentro de wine_pycaret/:")
    print("- reports/eda_wine_quality.html")
    print("- figures/class_balance.png")
    print("- leaderboard_compare_models.csv")
    print("- model_pipeline.pkl")
    print("- meta.json")
    print("\n➡ Próximo passo (Streamlit):")
    print("streamlit run app_streamlit.py")


if __name__ == "__main__":
    main()

Diretório base usado: D:\TreinaRecife\Python do Zero até a Análise de Dados\aprendizado\Códigos\wine_pycaret
[1/7] Carregando dataset...
Shape: (1143, 12)
[2/7] Gerando EDA (ydata-profiling)...


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 12/12 [00:00<?, ?it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

...

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Gradient Boosting Classifier,0.6735,0.8252,0.6735,0.6518,0.6541,0.4662,0.4697


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Extreme Gradient Boosting,0.6414,0.8108,0.6414,0.6232,0.6298,0.4255,0.4264


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Voting Classifier,0.6647,0.8434,0.6647,0.6403,0.6458,0.4542,0.4569


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Stacking Classifier,0.6910,0.8470,0.6910,0.6519,0.6683,0.4960,0.4991



✅ Melhor escolhido (holdout) = RandomForestClassifier | score=0.6739
Transformation Pipeline and Model Successfully Saved

✅ PIPELINE FINALIZADO COM SUCESSO
Estrutura gerada dentro de wine_pycaret/:
- reports/eda_wine_quality.html
- figures/class_balance.png
- leaderboard_compare_models.csv
- model_pipeline.pkl
- meta.json

➡ Próximo passo (Streamlit):
streamlit run app_streamlit.py
